In [2]:
import openeo

connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()


Authenticated using refresh token.


In [3]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [111.80, -7.40],
            [112.10, -7.40],
            [112.10, -7.70],
            [111.80, -7.70],
            [111.80, -7.40],
        ]
    ]
}

s5post = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-25", "2026-08-25"],
    spatial_extent={
        "west": 111.80,
        "south": -7.70,
        "east": 112.10,
        "north": -7.40
    },
    bands=["O3"],
)

# Agregasi harian agar tidak ada lebih dari satu data per hari
s5p_no2_daily = s5post.aggregate_temporal_period(reducer="mean", period="day")

# Agregasi spasial untuk menghasilkan rata-rata time series per AOI
s5p_no2_aoi = s5p_no2_daily.aggregate_spatial(reducer="mean", geometries=aoi)

In [5]:
job = s5post.execute_batch(title="CO3 di Nganjuk", outputfile="CO3DiNganjuk.nc")

0:00:00 Job 'j-2608280756434ac88d5feca129ff84c7': send 'start'
0:00:02 Job 'j-2608280756434ac88d5feca129ff84c7': queued (progress 0%)
0:00:08 Job 'j-2608280756434ac88d5feca129ff84c7': queued (progress 0%)
0:00:15 Job 'j-2608280756434ac88d5feca129ff84c7': queued (progress 0%)
0:00:23 Job 'j-2608280756434ac88d5feca129ff84c7': queued (progress 0%)
0:00:33 Job 'j-2608280756434ac88d5feca129ff84c7': queued (progress 0%)
0:00:46 Job 'j-2608280756434ac88d5feca129ff84c7': queued (progress 0%)
0:01:01 Job 'j-2608280756434ac88d5feca129ff84c7': queued (progress 0%)
0:01:21 Job 'j-2608280756434ac88d5feca129ff84c7': queued (progress 0%)
0:01:45 Job 'j-2608280756434ac88d5feca129ff84c7': running (progress N/A)
0:02:15 Job 'j-2608280756434ac88d5feca129ff84c7': running (progress N/A)
0:02:53 Job 'j-2608280756434ac88d5feca129ff84c7': running (progress N/A)
0:03:40 Job 'j-2608280756434ac88d5feca129ff84c7': running (progress N/A)
0:04:38 Job 'j-2608280756434ac88d5feca129ff84c7': running (progress N/A)
0:05

In [6]:
import numpy as np
import pandas as pd
import netCDF4

file_path = "CO3DiNganjuk.nc"
ds = netCDF4.Dataset(file_path)
# Ambil NO2
no2 = ds.variables["CO3"][:]

# Ambil Time
time = ds.variables["t"][:]

# Konversi waktu ke format tanggal
try:
    time_units = ds.variables["t"].units
    dates = netCDF4.num2date(time, units=time_units)
except Exception:
    dates = time  # fallback jika tidak ada units

        
new_dates = []
new_no2 = []

for i in range(len(dates)):
    new_date = dates[i].strftime('%Y-%m-%d')
    new_dates.append(new_date)
    new_no2.append(np.mean(no2[i]))

df = pd.DataFrame({
    "date": new_dates,
    "CO3": new_no2
})

# Simpan ke CSV
df.to_csv("CO3_Nganjuk_timeseries.csv", index=False)

KeyError: 'CO3'

In [11]:
import numpy as np
import pandas as pd
import netCDF4

file_path = "CO3DiNganjuk.nc"
ds = netCDF4.Dataset(file_path)
# Ambil NO2
no2 = ds.variables["O3"][:]

# Ambil Time
time = ds.variables["t"][:]

# Konversi waktu ke format tanggal
try:
    time_units = ds.variables["t"].units
    dates = netCDF4.num2date(time, units=time_units)
except Exception:
    dates = time  # fallback jika tidak ada units

        
new_dates = []
new_no2 = []

for i in range(len(dates)):
    new_date = dates[i].strftime('%Y-%m-%d')
    new_dates.append(new_date)
    new_no2.append(np.mean(no2[i]))

df = pd.DataFrame({
    "date": new_dates,
    "O3": new_no2
})

# Simpan ke CSV
df.to_csv("03_Nganjuk_timeseries.csv", index=False)

In [7]:
import pandas as pd

df = pd.read_csv("CO.csv")

# pastikan kolom tanggal valid
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# ambil hanya bulan dan tahun
df["date"] = df["date"].dt.strftime("%Y-%m-%d")

new_df = pd.DataFrame({
    "date": df['date'],
    "CO": df['CO']
})

new_df.to_csv("CO_Timeseries.csv")

In [5]:
print(df['date'])

0      2025-12-15T00:00:00.000Z
1      2025-12-10T00:00:00.000Z
2      2025-12-16T00:00:00.000Z
3      2025-12-17T00:00:00.000Z
4      2025-12-12T00:00:00.000Z
                 ...           
361    2026-08-15T00:00:00.000Z
362    2026-08-19T00:00:00.000Z
363    2026-08-22T00:00:00.000Z
364    2026-08-17T00:00:00.000Z
365    2026-08-20T00:00:00.000Z
Name: date, Length: 366, dtype: object
